# Brazil — Holt-Winters

## 1. Imports e configurações

In [ ]:
from pathlib import Path
import json, warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
warnings.filterwarnings("ignore")
CSV_PATH=Path("datasets/brazil.csv")
TARGET="goals_brazil"
TEST_SIZE=0.20
RANDOM_STATE=42

from statsmodels.tsa.holtwinters import ExponentialSmoothing
SEASONAL_PERIODS=[4,8,12]

## 2. Limpeza

In [ ]:
def load_and_clean(path):
    df=pd.read_csv(path).copy()
    df["date"]=pd.to_datetime(df["Date"],format="%d %b %Y",errors="coerce")
    df=df.sort_values("date",kind="stable").drop_duplicates("date",keep="first").reset_index(drop=True)
    score=df["Score"].astype("string").str.extract(r"^\s*(\d+)\s*-\s*(\d+)")
    df["home_goals"]=pd.to_numeric(score[0],errors="coerce")
    df["away_goals"]=pd.to_numeric(score[1],errors="coerce")
    df["is_home"]=df["Match"].str.startswith("Brazil v ").astype(int)
    df["opponent"]=np.where(df["is_home"].eq(1),df["Match"].str.replace(r"^Brazil v ","",regex=True),df["Match"].str.replace(r" v Brazil$","",regex=True))
    df["goals_brazil"]=np.where(df["is_home"].eq(1),df["home_goals"],df["away_goals"])
    df["goals_against"]=np.where(df["is_home"].eq(1),df["away_goals"],df["home_goals"])
    df["points"]=df["Result"].map({"W":3,"D":1,"L":0})
    df=df.dropna(subset=["date","Competition","opponent","goals_brazil","goals_against","points"]).copy()
    for c in ["goals_brazil","goals_against","points"]: df[c]=df[c].astype(float)
    return df.reset_index(drop=True)
raw=load_and_clean(CSV_PATH)
print("Linhas após limpeza:",len(raw)); display(raw.head())


## 3. Divisão temporal

In [ ]:
data=raw.dropna(subset=[TARGET]).reset_index(drop=True); split_idx=int(len(data)*(1-TEST_SIZE)); train=data.iloc[:split_idx].copy(); test=data.iloc[split_idx:].copy(); y_train=train[TARGET].to_numpy(float); y_test=test[TARGET].to_numpy(float); print("Treino:",len(train),"Teste:",len(test))

## 4. Busca de configuração

In [ ]:
configs=[{"trend":None,"damped_trend":False,"seasonal":None,"seasonal_periods":None},{"trend":"add","damped_trend":False,"seasonal":None,"seasonal_periods":None},{"trend":"add","damped_trend":True,"seasonal":None,"seasonal_periods":None}]
for m in SEASONAL_PERIODS:
    for trend in [None,"add"]:
        for damped in ([False] if trend is None else [False,True]): configs.append({"trend":trend,"damped_trend":damped,"seasonal":"add","seasonal_periods":m})
rows=[]; best_bic=np.inf; best_config=None
for cfg in configs:
    try:
        fit=ExponentialSmoothing(y_train,trend=cfg["trend"],damped_trend=cfg["damped_trend"],seasonal=cfg["seasonal"],seasonal_periods=cfg["seasonal_periods"],initialization_method="estimated").fit(optimized=True); bic=getattr(fit,"bic",np.nan); rows.append({**cfg,"AIC":getattr(fit,"aic",np.nan),"BIC":bic})
        if np.isfinite(bic) and bic<best_bic: best_bic,best_config=bic,cfg
    except Exception: pass
search_table=pd.DataFrame(rows).sort_values("BIC").reset_index(drop=True)
if best_config is None: raise RuntimeError("Nenhuma configuração convergiu")
print("Melhor configuração:",best_config,"BIC:",round(best_bic,2)); display(search_table.head(10))

## 5. Walk-forward final

In [ ]:
preds=[]
for i in range(len(y_test)):
    hist=np.concatenate([y_train,y_test[:i]])
    fit=ExponentialSmoothing(hist,trend=best_config["trend"],damped_trend=best_config["damped_trend"],seasonal=best_config["seasonal"],seasonal_periods=best_config["seasonal_periods"],initialization_method="estimated").fit(optimized=True)
    preds.append(float(np.ravel(fit.forecast(1))[0]))
preds=np.asarray(preds); results=pd.DataFrame({"date":test["date"].values,"real":y_test,"prediction":preds,"absolute_error":np.abs(y_test-preds)}); mae=mean_absolute_error(y_test,preds); print("MAE final walk-forward:",round(mae,4)); display(results.head(10))

## 6. Treino final e salvamento

In [ ]:
OUTPUT_DIR=Path("outputs_holt_winters_brazil"); OUTPUT_DIR.mkdir(exist_ok=True)
final=ExponentialSmoothing(data[TARGET].to_numpy(float),trend=best_config["trend"],damped_trend=best_config["damped_trend"],seasonal=best_config["seasonal"],seasonal_periods=best_config["seasonal_periods"],initialization_method="estimated").fit(optimized=True)
joblib.dump(final,OUTPUT_DIR/"holt_winters_brazil.joblib"); results.to_csv(OUTPUT_DIR/"walk_forward_predictions.csv",index=False)
with open(OUTPUT_DIR/"config.json","w",encoding="utf-8") as f: json.dump({"best_config":best_config,"BIC":float(best_bic),"MAE_walk_forward":float(mae)},f,ensure_ascii=False,indent=2,default=str)
print("Salvo em:",OUTPUT_DIR.resolve())